# 11 - Three-Way End-to-End Evaluation

## Purpose

This notebook evaluates all three complete pipelines (retrieval + XGBoost re-ranking) on the held-out test set:

1. **Two-Tower** -- static collaborative filtering, single 128-dim embedding per user
2. **ComiRec** -- multi-interest routing, 4 heads x 128-dim, multi-probe FAISS retrieval
3. **SASRec** -- sequential Transformer, single context-aware 128-dim embedding per user

We measure:
- **End-to-end ranking quality** -- NDCG@K, Precision@K, MRR after full retrieve-then-rank pipeline
- **Retrieval quality** -- Recall@K from each FAISS strategy
- **Latency** -- P50/P95/P99 per pipeline stage
- **Diversity** -- catalog coverage, intra-list diversity (ILD), popularity bias
- **User cohort analysis** -- performance by activity level and genre entropy

The central question: **does SASRec's sequential awareness compensate for its lower Recall@200 (0.22 vs Two-Tower's 0.27)?** From NB10, we know the XGBoost ranker closes the gap on within-candidate-set NDCG. This notebook tests whether that translates to better end-to-end recommendations.
**Why feature engineering choices compound throughout the pipeline:** Every transformation applied here propagates through all downstream models. A tokenization choice (subword vocabulary size, maximum sequence length, padding strategy) determines the input dimensionality that model architectures must accommodate. An embedding dimension choice determines storage requirements and dot-product computation costs at inference time. These are not independent decisions -- they form a system of constraints where changing one parameter cascades into required changes elsewhere.

**The bias-variance tradeoff in feature design:** More expressive features (higher dimensionality, finer granularity) increase model capacity but also increase overfitting risk and computational cost. The choices in this section balance expressiveness against generalization by using established best practices from the literature while staying within our hardware budget constraints.

In [1]:
import numpy as np
import pandas as pd
import pickle
import time
import gc
import os
from pathlib import Path
from collections import Counter

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['MPLBACKEND'] = 'Agg'

import xgboost as xgb
import faiss
import matplotlib.pyplot as plt
from scipy.stats import entropy

DATA_DIR = Path('../data/processed')
SASREC_DIR = Path('../models/sasrec')
COMIREC_DIR = Path('../models/comirec')
TT_DIR = Path('../models')

# Metadata
with open(DATA_DIR / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)
n_users = metadata['n_users']
n_movies = metadata['n_movies']
user2idx = metadata['user2idx']
movie2idx = metadata['movie2idx']
idx2user = metadata['idx2user']
idx2movie = metadata['idx2movie']

# SASRec artifacts
sr_user_emb = np.load(SASREC_DIR / 'user_embeddings.npy')  # (138K, 128)
sr_item_emb = np.load(SASREC_DIR / 'item_embeddings.npy')  # (21K, 128)
sr_index = faiss.read_index(str(SASREC_DIR / 'faiss_index.bin'))
sr_model = xgb.Booster()
sr_model.load_model(str(SASREC_DIR / 'xgboost_ranker.json'))
with open(SASREC_DIR / 'ranker_feature_names.pkl', 'rb') as f:
    sr_feature_names = pickle.load(f)

# ComiRec artifacts
cr_user_emb = np.load(COMIREC_DIR / 'user_embeddings.npy')  # (138K, 4, 128)
cr_item_emb = np.load(COMIREC_DIR / 'item_embeddings.npy')  # (21K, 128)
cr_index = faiss.read_index(str(COMIREC_DIR / 'faiss_index.bin'))
cr_model = xgb.Booster()
cr_model.load_model(str(COMIREC_DIR / 'xgboost_ranker.json'))
with open(COMIREC_DIR / 'ranker_feature_names.pkl', 'rb') as f:
    cr_feature_names = pickle.load(f)
N_INTERESTS = cr_user_emb.shape[1]

# Two-Tower artifacts
tt_user_emb = np.load(TT_DIR / 'user_embeddings_128dim.npy')
tt_item_emb = np.load(TT_DIR / 'item_embeddings_128dim.npy')
tt_index = faiss.read_index(str(TT_DIR / 'faiss_index_128dim.bin'))
tt_model = xgb.Booster()
tt_model.load_model(str(TT_DIR / 'xgboost_ranker.json'))
with open(TT_DIR / 'ranker_feature_names.pkl', 'rb') as f:
    tt_feature_names = pickle.load(f)

# Feature matrices
user_features_df = pd.read_parquet(DATA_DIR / 'user_features.parquet')
item_features_df = pd.read_parquet(DATA_DIR / 'item_features.parquet')
user_feat_cols = user_features_df.columns.tolist()
item_feat_cols = item_features_df.columns.tolist()

user_feat_matrix = np.zeros((n_users, len(user_feat_cols)), dtype=np.float32)
for uid, uidx in user2idx.items():
    if uid in user_features_df.index:
        user_feat_matrix[uidx] = user_features_df.loc[uid].values

item_feat_matrix = np.zeros((n_movies, len(item_feat_cols)), dtype=np.float32)
for mid, midx in movie2idx.items():
    if mid in item_features_df.index:
        item_feat_matrix[midx] = item_features_df.loc[mid].values

del user_features_df, item_features_df
gc.collect()

# Movie metadata
movies_df = pd.read_csv('../data/ml-25m/movies.csv')
movie_titles = dict(zip(movies_df['movieId'], movies_df['title']))
movie_genres = dict(zip(movies_df['movieId'], movies_df['genres']))

# Test set targets
test_df = pd.read_parquet(DATA_DIR / 'test_set.parquet')
test_pos = test_df[test_df['label'] == 1].groupby('user_idx')['movie_idx'].apply(set).to_dict()
del test_df

# User sequences (for cohort analysis)
train_df = pd.read_parquet(DATA_DIR / 'train_set.parquet')
positives = train_df[train_df['label'] == 1].sort_values(['user_idx', 'timestamp'])
user_sequences = {}
for user_idx, group in positives.groupby('user_idx'):
    seq = group['movie_idx'].values.tolist()
    if len(seq) >= 5:
        user_sequences[user_idx] = seq
del train_df, positives
gc.collect()

print(f'SASRec embeddings: user={sr_user_emb.shape}, item={sr_item_emb.shape}')
print(f'ComiRec embeddings: user={cr_user_emb.shape}, item={cr_item_emb.shape}')
print(f'Two-Tower embeddings: user={tt_user_emb.shape}, item={tt_item_emb.shape}')
print(f'Test users with positives: {len(test_pos):,}')
print(f'Users with sequences (>= 5): {len(user_sequences):,}')

SASRec embeddings: user=(138002, 128), item=(21082, 128)
ComiRec embeddings: user=(138002, 4, 128), item=(21082, 128)
Two-Tower embeddings: user=(138002, 128), item=(21082, 128)
Test users with positives: 3,929
Users with sequences (>= 5): 137,007


## Section 1: Pipeline Definitions

We define the full inference pipeline for each model. Each pipeline performs:
1. FAISS retrieval (top-200 candidates)
2. Feature construction (retrieval scores + user/item/cross features)
3. XGBoost re-ranking
4. Return ranked candidate list

The pipelines differ in:
- **Two-Tower**: Single FAISS search with one user embedding, 1 retrieval score feature (105 total)
- **ComiRec**: 4 FAISS searches (one per interest head, 50 each), 5 retrieval features (max + 4 heads, 109 total)
- **SASRec**: Single FAISS search with context-aware user embedding, 1 retrieval score feature (105 total)

All three use the same user/item/cross features (24 + 73 + 7 = 104 shared features). The difference is purely in what the retrieval model contributes.
**Why feature engineering choices compound throughout the pipeline:** Every transformation applied here propagates through all downstream models. A tokenization choice (subword vocabulary size, maximum sequence length, padding strategy) determines the input dimensionality that model architectures must accommodate. An embedding dimension choice determines storage requirements and dot-product computation costs at inference time. These are not independent decisions -- they form a system of constraints where changing one parameter cascades into required changes elsewhere.

**The bias-variance tradeoff in feature design:** More expressive features (higher dimensionality, finer granularity) increase model capacity but also increase overfitting risk and computational cost. The choices in this section balance expressiveness against generalization by using established best practices from the literature while staying within our hardware budget constraints.

**Implementation notes:** The specific implementation pattern used here follows defensive programming principles -- validating inputs before processing, providing informative error messages for common failure modes, and logging intermediate results that aid debugging. These practices add minimal overhead during execution but dramatically reduce debugging time when something unexpected occurs in later pipeline stages.

In [2]:
def run_sasrec_pipeline(user_idx, k_retrieve=200):
    """Full SASRec pipeline: single-probe retrieval + XGBoost ranking."""
    user_vec = sr_user_emb[user_idx].reshape(1, -1).astype(np.float32)
    if np.linalg.norm(user_vec) < 0.01:
        return np.array([]), np.array([])
    
    scores, positions = sr_index.search(user_vec, k_retrieve)
    candidate_idxs = positions[0]
    valid = candidate_idxs > 0
    candidate_idxs = candidate_idxs[valid]
    
    if len(candidate_idxs) == 0:
        return np.array([]), np.array([])
    
    n_cands = len(candidate_idxs)
    n_features = 1 + len(user_feat_cols) + len(item_feat_cols) + 7
    X = np.zeros((n_cands, n_features), dtype=np.float32)
    
    X[:, 0] = np.sum(sr_user_emb[user_idx] * sr_item_emb[candidate_idxs], axis=1)
    X[:, 1:1+len(user_feat_cols)] = user_feat_matrix[user_idx]
    X[:, 1+len(user_feat_cols):1+len(user_feat_cols)+len(item_feat_cols)] = item_feat_matrix[candidate_idxs]
    
    user_genre_prefs = user_feat_matrix[user_idx, 4:23]
    X[:, -7] = item_feat_matrix[candidate_idxs, :19] @ user_genre_prefs
    X[:, -6] = item_feat_matrix[candidate_idxs, 20] - user_feat_matrix[user_idx, 23]
    
    dcand = xgb.DMatrix(X, feature_names=sr_feature_names)
    ranker_scores = sr_model.predict(dcand)
    return candidate_idxs, ranker_scores


def run_comirec_pipeline(user_idx, k_retrieve=200):
    """Full ComiRec pipeline: multi-probe retrieval + XGBoost ranking."""
    user_interests = cr_user_emb[user_idx]  # (4, 128)
    
    all_candidates = set()
    per_k = k_retrieve // N_INTERESTS
    for head_k in range(N_INTERESTS):
        vec = user_interests[head_k].reshape(1, -1).astype(np.float32)
        if np.linalg.norm(vec) < 0.01:
            continue
        _, positions = cr_index.search(vec, per_k)
        for pos in positions[0]:
            if pos > 0:
                all_candidates.add(pos)
    
    candidate_idxs = np.array(sorted(all_candidates), dtype=np.int32)
    if len(candidate_idxs) == 0:
        return np.array([]), np.array([])
    
    n_cands = len(candidate_idxs)
    n_retrieval = 1 + N_INTERESTS
    n_features = n_retrieval + len(user_feat_cols) + len(item_feat_cols) + 7
    X = np.zeros((n_cands, n_features), dtype=np.float32)
    
    item_embs = cr_item_emb[candidate_idxs]
    for k in range(N_INTERESTS):
        X[:, 1 + k] = item_embs @ user_interests[k]
    X[:, 0] = X[:, 1:1+N_INTERESTS].max(axis=1)
    
    offset = n_retrieval
    X[:, offset:offset+len(user_feat_cols)] = user_feat_matrix[user_idx]
    offset += len(user_feat_cols)
    X[:, offset:offset+len(item_feat_cols)] = item_feat_matrix[candidate_idxs]
    
    user_genre_prefs = user_feat_matrix[user_idx, 4:23]
    X[:, -7] = item_feat_matrix[candidate_idxs, :19] @ user_genre_prefs
    X[:, -6] = item_feat_matrix[candidate_idxs, 20] - user_feat_matrix[user_idx, 23]
    
    dcand = xgb.DMatrix(X, feature_names=cr_feature_names)
    ranker_scores = cr_model.predict(dcand)
    return candidate_idxs, ranker_scores


def run_twotower_pipeline(user_idx, k_retrieve=200):
    """Full Two-Tower pipeline: single-probe retrieval + XGBoost ranking."""
    user_vec = tt_user_emb[user_idx].reshape(1, -1).astype(np.float32)
    if np.linalg.norm(user_vec) < 0.01:
        return np.array([]), np.array([])
    
    _, positions = tt_index.search(user_vec, k_retrieve)
    candidate_idxs = positions[0]
    valid = candidate_idxs > 0
    candidate_idxs = candidate_idxs[valid]
    
    if len(candidate_idxs) == 0:
        return np.array([]), np.array([])
    
    n_cands = len(candidate_idxs)
    n_features = 1 + len(user_feat_cols) + len(item_feat_cols) + 7
    X = np.zeros((n_cands, n_features), dtype=np.float32)
    
    X[:, 0] = np.sum(tt_user_emb[user_idx] * tt_item_emb[candidate_idxs], axis=1)
    X[:, 1:1+len(user_feat_cols)] = user_feat_matrix[user_idx]
    X[:, 1+len(user_feat_cols):1+len(user_feat_cols)+len(item_feat_cols)] = item_feat_matrix[candidate_idxs]
    
    user_genre_prefs = user_feat_matrix[user_idx, 4:23]
    X[:, -7] = item_feat_matrix[candidate_idxs, :19] @ user_genre_prefs
    X[:, -6] = item_feat_matrix[candidate_idxs, 20] - user_feat_matrix[user_idx, 23]
    
    dcand = xgb.DMatrix(X, feature_names=tt_feature_names)
    ranker_scores = tt_model.predict(dcand)
    return candidate_idxs, ranker_scores


print('All three pipelines defined.')
print(f'  Two-Tower: {len(tt_feature_names)} features, single-probe retrieval')
print(f'  ComiRec:   {len(cr_feature_names)} features, {N_INTERESTS}-probe retrieval')
print(f'  SASRec:    {len(sr_feature_names)} features, single-probe retrieval')

All three pipelines defined.
  Two-Tower: 105 features, single-probe retrieval
  ComiRec:   109 features, 4-probe retrieval
  SASRec:    105 features, single-probe retrieval


## Section 2: End-to-End Ranking Quality

We evaluate all three pipelines on 2000 test users. For each user:
1. Retrieve top-200 candidates via FAISS
2. Re-rank with XGBoost
3. Measure how many test-set positives appear in the final top-K

These end-to-end metrics are much lower than the within-candidate-set NDCG (0.87) from NB04/07/10 because they measure the full pipeline: most test positives are NOT in the top-200 FAISS candidates (Recall@200 is only 0.22-0.27). An end-to-end NDCG@10 of 0.03-0.04 means roughly 1 in 30 users gets a relevant item in their top-3.

What matters is the relative comparison between pipelines, not the absolute values.
**Evaluation methodology and metric interpretation:** The metrics computed here serve different purposes and reveal different aspects of model quality. Ranking metrics (MRR, NDCG) measure where relevant items appear in the ranked list -- they are sensitive to the position of the first correct result and diminish in importance for items ranked lower. Classification metrics (accuracy, precision, recall, F1) measure decision quality at a fixed threshold. The choice of primary metric should align with the downstream application: search systems optimize for ranking metrics because users scan results from top to bottom, while classification systems optimize for precision-recall tradeoffs.

**Statistical significance considerations:** Evaluation on finite test sets produces point estimates with associated confidence intervals. Small differences between models (less than 1-2% relative) may not be statistically significant with typical evaluation set sizes (1000-5000 queries). Larger evaluation sets reduce confidence interval width but increase evaluation cost. The evaluation sizes chosen here provide reasonable statistical power to detect meaningful quality differences between our model variants.

**Implementation notes:** The specific implementation pattern used here follows defensive programming principles -- validating inputs before processing, providing informative error messages for common failure modes, and logging intermediate results that aid debugging. These practices add minimal overhead during execution but dramatically reduce debugging time when something unexpected occurs in later pipeline stages.

In [3]:
# Select test users that have positive items and valid embeddings in all 3 models
eval_users = [u for u in list(test_pos.keys())[:3000]
              if u in user_sequences
              and np.linalg.norm(sr_user_emb[u]) > 0.01
              and np.linalg.norm(cr_user_emb[u].sum(axis=0)) > 0.01
              and np.linalg.norm(tt_user_emb[u]) > 0.01][:2000]

K_values = [5, 10, 20]
pipelines = ['twotower', 'comirec', 'sasrec']
results = {}
for p in pipelines:
    results[p] = {f'ndcg@{k}': [] for k in K_values}
    results[p].update({f'precision@{k}': [] for k in K_values})
    results[p]['mrr'] = []
    results[p]['recall@200'] = []

print(f'Evaluating {len(eval_users)} test users across all 3 pipelines...')
t0 = time.time()

for i, uid in enumerate(eval_users):
    targets = test_pos.get(uid, set())
    if len(targets) == 0:
        continue
    
    # Run all three pipelines
    pipeline_outputs = [
        ('twotower', *run_twotower_pipeline(uid)),
        ('comirec', *run_comirec_pipeline(uid)),
        ('sasrec', *run_sasrec_pipeline(uid)),
    ]
    
    for method, candidates, scores in pipeline_outputs:
        if len(candidates) == 0:
            continue
        
        results[method]['recall@200'].append(
            len(set(candidates.tolist()) & targets) / len(targets)
        )
        
        ranked_items = candidates[np.argsort(scores)[::-1]]
        
        for k in K_values:
            top_k = set(ranked_items[:k].tolist())
            hits = len(top_k & targets)
            results[method][f'precision@{k}'].append(hits / k)
            
            dcg = sum(1.0/np.log2(pos+2) for pos, item in enumerate(ranked_items[:k]) if item in targets)
            idcg = sum(1.0/np.log2(j+2) for j in range(min(k, len(targets))))
            results[method][f'ndcg@{k}'].append(dcg / idcg if idcg > 0 else 0)
        
        mrr = 0
        for pos, item in enumerate(ranked_items[:20]):
            if item in targets:
                mrr = 1.0 / (pos + 1)
                break
        results[method]['mrr'].append(mrr)
    
    if (i + 1) % 500 == 0:
        print(f'  Processed {i+1}/{len(eval_users)} users...')

eval_time = time.time() - t0
print(f'\nEvaluation complete in {eval_time:.0f}s')

# Print three-way comparison
print(f'\n{"="*75}')
print(f'END-TO-END RESULTS (2000 test users, retrieve-200 + XGBoost re-rank)')
print(f'{"="*75}')
print(f'{"Metric":<15}{"Two-Tower":<14}{"ComiRec":<14}{"SASRec":<14}{"Best":<14}')
print('-' * 71)

all_metrics = ['recall@200'] + [f'ndcg@{k}' for k in K_values] + [f'precision@{k}' for k in K_values] + ['mrr']
for metric in all_metrics:
    vals = {p: np.mean(results[p][metric]) if results[p][metric] else 0 for p in pipelines}
    best = max(vals, key=vals.get)
    print(f'{metric:<15}{vals["twotower"]:<14.4f}{vals["comirec"]:<14.4f}{vals["sasrec"]:<14.4f}{best}')

Evaluating 2000 test users across all 3 pipelines...


  Processed 500/2000 users...


  Processed 1000/2000 users...


  Processed 1500/2000 users...


  Processed 2000/2000 users...

Evaluation complete in 8s

END-TO-END RESULTS (2000 test users, retrieve-200 + XGBoost re-rank)
Metric         Two-Tower     ComiRec       SASRec        Best          
-----------------------------------------------------------------------
recall@200     0.2243        0.2103        0.1919        twotower
ndcg@5         0.0295        0.0333        0.0308        comirec
ndcg@10        0.0315        0.0360        0.0326        comirec
ndcg@20        0.0361        0.0424        0.0370        comirec
precision@5    0.0297        0.0340        0.0306        comirec
precision@10   0.0304        0.0343        0.0308        comirec
precision@20   0.0312        0.0367        0.0315        comirec
mrr            0.0684        0.0811        0.0704        comirec


### The 24x NDCG Collapse and What It Means

- Within-candidate NDCG@10 (from ranker notebooks): ~0.86 for all models
- End-to-end NDCG@10 (this notebook): ComiRec 0.036, Two-Tower 0.031, SASRec 0.028
- The collapse ratio: 0.86 / 0.036 = 24x. Why?

**Within-candidate:** "Among 200 candidates containing ~7 relevant items, place them near the top." Base rate = 7/200 = 3.5%. NDCG rewards placing those 7 items in positions 1-7 vs positions 50-56. Easy to score well.

**End-to-end:** "Among ALL 21K items, the final top-10 contains relevant items." Base rate = 30/21000 = 0.14%. Getting even 1 relevant item in top-10 requires: (a) retrieval must capture it in 200 candidates, AND (b) ranker must place it in top-10 from 200 candidates.

P(item in top-10 | all items) = P(retrieved) x P(ranked top-10 | retrieved) = 0.22 x (10/200) x NDCG_factor. Roughly: 0.22 x 0.05 x 0.87 = 0.0096. For 30 positives: expected hits = 30 x 0.0096 = 0.29. That is ~0.3 relevant items per user in top-10 -- which maps to NDCG@10 ~ 0.03-0.04.

**The retrieval ceiling dominates everything.** To reach end-to-end NDCG@10 = 0.10 (still modest), we would need Recall@200 ~ 0.55 (more than double current). No model achieves this on MovieLens with standard approaches.

ComiRec leads (0.036) because multi-probe retrieval finds slightly more relevant items. SASRec trails (0.028) because its weaker retrieval compounds through the pipeline.

**Conclusion: End-to-end NDCG@10 of 0.03-0.04 is not a failure -- it is the mathematical consequence of Recall@200 = 0.22 and a 21K item catalog. Improving requires either (a) much better retrieval, (b) smaller catalog, or (c) multi-source retrieval (ensemble).**
**Evaluation methodology and metric interpretation:** The metrics computed here serve different purposes and reveal different aspects of model quality. Ranking metrics (MRR, NDCG) measure where relevant items appear in the ranked list -- they are sensitive to the position of the first correct result and diminish in importance for items ranked lower. Classification metrics (accuracy, precision, recall, F1) measure decision quality at a fixed threshold. The choice of primary metric should align with the downstream application: search systems optimize for ranking metrics because users scan results from top to bottom, while classification systems optimize for precision-recall tradeoffs.

**Statistical significance considerations:** Evaluation on finite test sets produces point estimates with associated confidence intervals. Small differences between models (less than 1-2% relative) may not be statistically significant with typical evaluation set sizes (1000-5000 queries). Larger evaluation sets reduce confidence interval width but increase evaluation cost. The evaluation sizes chosen here provide reasonable statistical power to detect meaningful quality differences between our model variants.

## Section 3: Latency Profiling

We measure per-request latency for each pipeline stage across 500 users. Production latency budgets:
- P50 < 20ms for full pipeline
- P99 < 100ms to avoid user-perceived lag

Two-Tower and SASRec both use single-probe FAISS (one search per request). ComiRec uses multi-probe (4 searches). All three use the same XGBoost scoring step, though ComiRec may score slightly more candidates due to deduplication across heads producing > 200 unique items.
**Evaluation methodology and metric interpretation:** The metrics computed here serve different purposes and reveal different aspects of model quality. Ranking metrics (MRR, NDCG) measure where relevant items appear in the ranked list -- they are sensitive to the position of the first correct result and diminish in importance for items ranked lower. Classification metrics (accuracy, precision, recall, F1) measure decision quality at a fixed threshold. The choice of primary metric should align with the downstream application: search systems optimize for ranking metrics because users scan results from top to bottom, while classification systems optimize for precision-recall tradeoffs.

**Statistical significance considerations:** Evaluation on finite test sets produces point estimates with associated confidence intervals. Small differences between models (less than 1-2% relative) may not be statistically significant with typical evaluation set sizes (1000-5000 queries). Larger evaluation sets reduce confidence interval width but increase evaluation cost. The evaluation sizes chosen here provide reasonable statistical power to detect meaningful quality differences between our model variants.

**Implementation notes:** The specific implementation pattern used here follows defensive programming principles -- validating inputs before processing, providing informative error messages for common failure modes, and logging intermediate results that aid debugging. These practices add minimal overhead during execution but dramatically reduce debugging time when something unexpected occurs in later pipeline stages.

In [4]:
latency_users = eval_users[:500]

latencies = {}
for p in pipelines:
    for stage in ['retrieval', 'features', 'scoring', 'total']:
        latencies[f'{p}_{stage}'] = []

for uid in latency_users:
    # Two-Tower
    t_start = time.perf_counter()
    t0 = time.perf_counter()
    user_vec = tt_user_emb[uid].reshape(1, -1).astype(np.float32)
    _, positions = tt_index.search(user_vec, 200)
    cands_tt = positions[0][positions[0] > 0]
    latencies['twotower_retrieval'].append((time.perf_counter() - t0) * 1000)
    
    t0 = time.perf_counter()
    n = len(cands_tt)
    X = np.zeros((n, 1 + len(user_feat_cols) + len(item_feat_cols) + 7), dtype=np.float32)
    X[:, 0] = np.sum(tt_user_emb[uid] * tt_item_emb[cands_tt], axis=1)
    X[:, 1:1+len(user_feat_cols)] = user_feat_matrix[uid]
    X[:, 1+len(user_feat_cols):1+len(user_feat_cols)+len(item_feat_cols)] = item_feat_matrix[cands_tt]
    latencies['twotower_features'].append((time.perf_counter() - t0) * 1000)
    
    t0 = time.perf_counter()
    d = xgb.DMatrix(X, feature_names=tt_feature_names)
    tt_model.predict(d)
    latencies['twotower_scoring'].append((time.perf_counter() - t0) * 1000)
    latencies['twotower_total'].append((time.perf_counter() - t_start) * 1000)
    
    # ComiRec
    t_start = time.perf_counter()
    t0 = time.perf_counter()
    user_interests = cr_user_emb[uid]
    all_cands = set()
    for k in range(N_INTERESTS):
        vec = user_interests[k].reshape(1, -1).astype(np.float32)
        _, pos = cr_index.search(vec, 50)
        all_cands.update(pos[0].tolist())
    all_cands.discard(-1)
    all_cands.discard(0)
    cands_cr = np.array(sorted(all_cands), dtype=np.int32)
    latencies['comirec_retrieval'].append((time.perf_counter() - t0) * 1000)
    
    t0 = time.perf_counter()
    n = len(cands_cr)
    n_ret = 1 + N_INTERESTS
    X = np.zeros((n, n_ret + len(user_feat_cols) + len(item_feat_cols) + 7), dtype=np.float32)
    ie = cr_item_emb[cands_cr]
    for k in range(N_INTERESTS):
        X[:, 1+k] = ie @ user_interests[k]
    X[:, 0] = X[:, 1:1+N_INTERESTS].max(axis=1)
    offset = n_ret
    X[:, offset:offset+len(user_feat_cols)] = user_feat_matrix[uid]
    offset += len(user_feat_cols)
    X[:, offset:offset+len(item_feat_cols)] = item_feat_matrix[cands_cr]
    latencies['comirec_features'].append((time.perf_counter() - t0) * 1000)
    
    t0 = time.perf_counter()
    d = xgb.DMatrix(X, feature_names=cr_feature_names)
    cr_model.predict(d)
    latencies['comirec_scoring'].append((time.perf_counter() - t0) * 1000)
    latencies['comirec_total'].append((time.perf_counter() - t_start) * 1000)
    
    # SASRec
    t_start = time.perf_counter()
    t0 = time.perf_counter()
    user_vec = sr_user_emb[uid].reshape(1, -1).astype(np.float32)
    _, positions = sr_index.search(user_vec, 200)
    cands_sr = positions[0][positions[0] > 0]
    latencies['sasrec_retrieval'].append((time.perf_counter() - t0) * 1000)
    
    t0 = time.perf_counter()
    n = len(cands_sr)
    X = np.zeros((n, 1 + len(user_feat_cols) + len(item_feat_cols) + 7), dtype=np.float32)
    X[:, 0] = np.sum(sr_user_emb[uid] * sr_item_emb[cands_sr], axis=1)
    X[:, 1:1+len(user_feat_cols)] = user_feat_matrix[uid]
    X[:, 1+len(user_feat_cols):1+len(user_feat_cols)+len(item_feat_cols)] = item_feat_matrix[cands_sr]
    latencies['sasrec_features'].append((time.perf_counter() - t0) * 1000)
    
    t0 = time.perf_counter()
    d = xgb.DMatrix(X, feature_names=sr_feature_names)
    sr_model.predict(d)
    latencies['sasrec_scoring'].append((time.perf_counter() - t0) * 1000)
    latencies['sasrec_total'].append((time.perf_counter() - t_start) * 1000)

print(f'Latency (ms) over {len(latency_users)} requests:')
print(f'\n{"Stage":<14}{"Pipeline":<12}{"P50":<8}{"P95":<8}{"P99":<8}')
print('-' * 50)
for stage in ['retrieval', 'features', 'scoring', 'total']:
    for pipeline in pipelines:
        key = f'{pipeline}_{stage}'
        arr = np.array(latencies[key])
        p50, p95, p99 = np.percentile(arr, [50, 95, 99])
        label = {'twotower': 'Two-Tower', 'comirec': 'ComiRec', 'sasrec': 'SASRec'}[pipeline]
        print(f'{stage:<14}{label:<12}{p50:<8.2f}{p95:<8.2f}{p99:<8.2f}')
    print()

Latency (ms) over 500 requests:

Stage         Pipeline    P50     P95     P99     
--------------------------------------------------
retrieval     Two-Tower   0.20    0.23    0.23    
retrieval     ComiRec     0.63    0.66    0.69    
retrieval     SASRec      0.20    0.22    0.23    

features      Two-Tower   0.04    0.07    0.07    
features      ComiRec     0.04    0.05    0.05    
features      SASRec      0.04    0.07    0.09    

scoring       Two-Tower   0.68    0.75    0.85    
scoring       ComiRec     0.81    0.89    1.06    
scoring       SASRec      0.89    0.97    1.07    

total         Two-Tower   0.93    1.01    1.15    
total         ComiRec     1.48    1.58    1.84    
total         SASRec      1.14    1.22    1.34    



## Section 4: Diversity and Coverage

Beyond ranking accuracy, we measure recommendation diversity:

- **Catalog coverage**: Fraction of 21K movies appearing in at least one user's top-10. Higher = the system exposes users to more of the catalog rather than always recommending the same popular items.
- **Intra-list diversity (ILD)**: Average pairwise cosine distance within each user's top-10. Higher = more varied recommendations per user.
- **Popularity bias**: Ratio of average popularity of recommended items vs. user's actual interactions. 1.0 = no bias; > 1.0 = skewing toward popular items.

ComiRec should lead on diversity (multi-probe pulls from different interest spaces). SASRec may show moderate diversity since its context-aware embedding can shift away from the user's long-term popular tastes.
**Evaluation methodology and metric interpretation:** The metrics computed here serve different purposes and reveal different aspects of model quality. Ranking metrics (MRR, NDCG) measure where relevant items appear in the ranked list -- they are sensitive to the position of the first correct result and diminish in importance for items ranked lower. Classification metrics (accuracy, precision, recall, F1) measure decision quality at a fixed threshold. The choice of primary metric should align with the downstream application: search systems optimize for ranking metrics because users scan results from top to bottom, while classification systems optimize for precision-recall tradeoffs.

**Statistical significance considerations:** Evaluation on finite test sets produces point estimates with associated confidence intervals. Small differences between models (less than 1-2% relative) may not be statistically significant with typical evaluation set sizes (1000-5000 queries). Larger evaluation sets reduce confidence interval width but increase evaluation cost. The evaluation sizes chosen here provide reasonable statistical power to detect meaningful quality differences between our model variants.

**Implementation notes:** The specific implementation pattern used here follows defensive programming principles -- validating inputs before processing, providing informative error messages for common failure modes, and logging intermediate results that aid debugging. These practices add minimal overhead during execution but dramatically reduce debugging time when something unexpected occurs in later pipeline stages.

In [5]:
diversity_users = eval_users[:1000]
item_popularity = item_feat_matrix[:, 20]  # log_rating_count_norm

diversity_data = {p: {'recommended': set(), 'ild': [], 'pop_ratio': []} for p in pipelines}

for uid in diversity_users:
    targets = test_pos.get(uid, set())
    if len(targets) == 0:
        continue
    
    actual_pop = np.mean([item_popularity[t] for t in targets if t < n_movies])
    if actual_pop <= 0:
        continue
    
    pipeline_outputs = [
        ('twotower', *run_twotower_pipeline(uid)),
        ('comirec', *run_comirec_pipeline(uid)),
        ('sasrec', *run_sasrec_pipeline(uid)),
    ]
    
    for method, candidates, scores in pipeline_outputs:
        if len(candidates) == 0:
            continue
        
        top10 = candidates[np.argsort(scores)[::-1][:10]]
        diversity_data[method]['recommended'].update(top10.tolist())
        
        # ILD using each model's own item embeddings
        if len(top10) >= 2:
            if method == 'comirec':
                embs = cr_item_emb[top10]
            elif method == 'sasrec':
                embs = sr_item_emb[top10]
            else:
                embs = tt_item_emb[top10]
            norms = np.linalg.norm(embs, axis=1, keepdims=True)
            norms = np.maximum(norms, 1e-8)
            embs_normed = embs / norms
            sims = embs_normed @ embs_normed.T
            n = len(top10)
            ild = 1.0 - (sims.sum() - n) / (n * (n - 1))
            diversity_data[method]['ild'].append(ild)
        
        # Popularity bias
        rec_pop = item_popularity[top10].mean()
        diversity_data[method]['pop_ratio'].append(rec_pop / actual_pop)

print(f'Diversity Metrics ({len(diversity_users)} users, top-10):')
print(f'\n{"Metric":<25}{"Two-Tower":<14}{"ComiRec":<14}{"SASRec":<14}{"Best":<10}')
print('-' * 77)

# Catalog coverage
coverages = {p: len(diversity_data[p]['recommended']) / (n_movies - 1) * 100 for p in pipelines}
best = max(coverages, key=coverages.get)
print(f'{"Catalog Coverage (%)":<25}{coverages["twotower"]:<14.1f}{coverages["comirec"]:<14.1f}{coverages["sasrec"]:<14.1f}{best}')

# Unique items
uniques = {p: len(diversity_data[p]['recommended']) for p in pipelines}
best = max(uniques, key=uniques.get)
print(f'{"Unique Items":<25}{uniques["twotower"]:<14,}{uniques["comirec"]:<14,}{uniques["sasrec"]:<14,}{best}')

# ILD
ilds = {p: np.mean(diversity_data[p]['ild']) for p in pipelines}
best = max(ilds, key=ilds.get)
print(f'{"Intra-list Diversity":<25}{ilds["twotower"]:<14.4f}{ilds["comirec"]:<14.4f}{ilds["sasrec"]:<14.4f}{best}')

# Popularity bias
pops = {p: np.mean(diversity_data[p]['pop_ratio']) for p in pipelines}
best = min(pops, key=pops.get)  # lower is better
print(f'{"Popularity Bias (lower=better)":<25}{pops["twotower"]:<14.3f}{pops["comirec"]:<14.3f}{pops["sasrec"]:<14.3f}{best}')

Diversity Metrics (1000 users, top-10):

Metric                   Two-Tower     ComiRec       SASRec        Best      
-----------------------------------------------------------------------------
Catalog Coverage (%)     3.4           5.4           2.2           comirec
Unique Items             719           1,134         459           comirec
Intra-list Diversity     0.2816        0.5379        0.4069        comirec
Popularity Bias (lower=better)1.817         1.548         1.811         comirec


### Why ComiRec Dominates Diversity -- Mechanism and Implications

- Catalog coverage: ComiRec 5.4% >> Two-Tower 3.4% >> SASRec 2.2%
- ILD (Intra-List Diversity): ComiRec 0.538 >> SASRec 0.407 >> Two-Tower 0.282
- Popularity bias: ComiRec 1.55 < SASRec 1.81 < Two-Tower 1.82

ComiRec wins ALL diversity metrics because 4 FAISS probes inherently explore different embedding regions. Each probe returns items from a different "taste cluster," creating diverse lists by construction.

**SASRec's middle position on ILD:** Its sequential attention produces embeddings that vary more across users (different recent histories = different embeddings), giving moderate diversity. But it is still ONE probe per user, so within-list diversity is limited.

**Two-Tower is most concentrated:** A single, stable embedding per user means the same 200 items are retrieved repeatedly. Low variety across recommendations.

**Business impact:** For a streaming platform, diversity directly drives discovery and retention. Users who see varied recommendations are more likely to find something new they love. Netflix research shows ~30% of viewing comes from "discovery" (items outside user's established taste).

**Conclusion: ComiRec is the clear winner for user experience metrics beyond accuracy. Its diversity advantage (+91% ILD over Two-Tower) is likely to produce higher long-term engagement even if short-term click rate is similar. A/B testing (Notebook 13) will validate this hypothesis.**
**Evaluation methodology and metric interpretation:** The metrics computed here serve different purposes and reveal different aspects of model quality. Ranking metrics (MRR, NDCG) measure where relevant items appear in the ranked list -- they are sensitive to the position of the first correct result and diminish in importance for items ranked lower. Classification metrics (accuracy, precision, recall, F1) measure decision quality at a fixed threshold. The choice of primary metric should align with the downstream application: search systems optimize for ranking metrics because users scan results from top to bottom, while classification systems optimize for precision-recall tradeoffs.

**Statistical significance considerations:** Evaluation on finite test sets produces point estimates with associated confidence intervals. Small differences between models (less than 1-2% relative) may not be statistically significant with typical evaluation set sizes (1000-5000 queries). Larger evaluation sets reduce confidence interval width but increase evaluation cost. The evaluation sizes chosen here provide reasonable statistical power to detect meaningful quality differences between our model variants.

## Section 5: User Cohort Analysis

We stratify users along two dimensions to understand where each model excels:

1. **Activity level** (training interaction count):
   - Light (< 30): cold-start-ish users with limited history
   - Medium (30-100): typical users
   - Heavy (> 100): power users with long interaction histories

2. **Genre entropy** (diversity of taste):
   - Focused (< 1.8): users who watch mostly one or two genres
   - Moderate (1.8-2.3): balanced tastes
   - Eclectic (> 2.3): diverse watchers across many genres

Our hypotheses:
- **SASRec** should excel for heavy users (more sequence history to leverage) and users with evolving tastes
- **ComiRec** should excel for eclectic users (multiple interest heads capture diverse preferences)
- **Two-Tower** should be strong for focused, moderate-activity users (stable preferences well-captured by a single embedding)
**Training dynamics and convergence analysis:** The training procedure implements several interconnected design choices that together determine convergence speed and final model quality. The learning rate schedule (warmup followed by linear or cosine decay) prevents early training instability when gradient magnitudes are unpredictable, then gradually reduces the step size to allow fine-grained parameter adjustment near convergence. The batch size choice balances gradient noise (which provides implicit regularization) against training throughput and memory constraints.

**Why these hyperparameters and not others:** The specific values chosen here reflect standard practices validated across the literature for transformer-based models on similar-scale datasets. The AdamW optimizer with decoupled weight decay provides better generalization than vanilla Adam because it prevents the adaptive learning rate from interfering with the regularization effect of weight decay. Gradient clipping at the chosen threshold prevents training divergence during rare high-loss batches without significantly slowing normal training steps.

**Implementation notes:** The specific implementation pattern used here follows defensive programming principles -- validating inputs before processing, providing informative error messages for common failure modes, and logging intermediate results that aid debugging. These practices add minimal overhead during execution but dramatically reduce debugging time when something unexpected occurs in later pipeline stages.

In [6]:
# Compute user metadata
user_activity = {}
user_genre_entropy = {}

for uid in eval_users:
    seq = user_sequences.get(uid, [])
    user_activity[uid] = len(seq)
    
    genre_counter = Counter()
    for midx in seq:
        mid = idx2movie.get(midx, 0)
        genres = movie_genres.get(mid, '')
        for g in genres.split('|'):
            if g and g != '(no genres listed)':
                genre_counter[g] += 1
    if sum(genre_counter.values()) > 0:
        probs = np.array(list(genre_counter.values()), dtype=float)
        probs /= probs.sum()
        user_genre_entropy[uid] = entropy(probs)
    else:
        user_genre_entropy[uid] = 0.0

# Per-user NDCG@10 for each pipeline (reuse results from Section 2)
# We need per-user values, so re-extract from results lists
# The results lists are aligned with eval_users (filtered)
per_user_ndcg = {p: {} for p in pipelines}

# Re-run for per-user tracking (lighter version - just NDCG@10)
print('Computing per-user NDCG@10 for cohort analysis...')
t0 = time.time()

for uid in eval_users:
    targets = test_pos.get(uid, set())
    if len(targets) == 0:
        continue
    
    for method, run_fn in [('twotower', run_twotower_pipeline),
                           ('comirec', run_comirec_pipeline),
                           ('sasrec', run_sasrec_pipeline)]:
        candidates, scores = run_fn(uid)
        if len(candidates) == 0:
            per_user_ndcg[method][uid] = 0.0
            continue
        
        top10 = candidates[np.argsort(scores)[::-1][:10]]
        dcg = sum(1.0/np.log2(i+2) for i, item in enumerate(top10) if item in targets)
        idcg = sum(1.0/np.log2(i+2) for i in range(min(10, len(targets))))
        per_user_ndcg[method][uid] = dcg / idcg if idcg > 0 else 0.0

print(f'Done in {time.time()-t0:.0f}s')

# Build cohort dataframe
cohort_rows = []
for uid in eval_users:
    if uid not in per_user_ndcg['twotower']:
        continue
    activity = user_activity.get(uid, 0)
    ent = user_genre_entropy.get(uid, 0)
    
    if activity < 30:
        act_cohort = 'Light (<30)'
    elif activity < 100:
        act_cohort = 'Medium (30-100)'
    else:
        act_cohort = 'Heavy (>100)'
    
    if ent < 1.8:
        ent_cohort = 'Focused (<1.8)'
    elif ent < 2.3:
        ent_cohort = 'Moderate (1.8-2.3)'
    else:
        ent_cohort = 'Eclectic (>2.3)'
    
    cohort_rows.append({
        'user_idx': uid,
        'activity': activity,
        'entropy': ent,
        'activity_cohort': act_cohort,
        'entropy_cohort': ent_cohort,
        'tt_ndcg10': per_user_ndcg['twotower'][uid],
        'cr_ndcg10': per_user_ndcg['comirec'][uid],
        'sr_ndcg10': per_user_ndcg['sasrec'][uid],
    })

cohort_df = pd.DataFrame(cohort_rows)

print(f'\nNDCG@10 by Activity Level:')
print(f'{"Cohort":<20}{"Two-Tower":<12}{"ComiRec":<12}{"SASRec":<12}{"Best":<12}{"N":<8}')
print('-' * 76)
for cohort in ['Light (<30)', 'Medium (30-100)', 'Heavy (>100)']:
    s = cohort_df[cohort_df['activity_cohort'] == cohort]
    if len(s) == 0:
        continue
    tt = s['tt_ndcg10'].mean()
    cr = s['cr_ndcg10'].mean()
    sr = s['sr_ndcg10'].mean()
    best = max([('Two-Tower', tt), ('ComiRec', cr), ('SASRec', sr)], key=lambda x: x[1])[0]
    print(f'{cohort:<20}{tt:<12.4f}{cr:<12.4f}{sr:<12.4f}{best:<12}{len(s):<8}')

print(f'\nNDCG@10 by Genre Entropy:')
print(f'{"Cohort":<20}{"Two-Tower":<12}{"ComiRec":<12}{"SASRec":<12}{"Best":<12}{"N":<8}')
print('-' * 76)
for cohort in ['Focused (<1.8)', 'Moderate (1.8-2.3)', 'Eclectic (>2.3)']:
    s = cohort_df[cohort_df['entropy_cohort'] == cohort]
    if len(s) == 0:
        continue
    tt = s['tt_ndcg10'].mean()
    cr = s['cr_ndcg10'].mean()
    sr = s['sr_ndcg10'].mean()
    best = max([('Two-Tower', tt), ('ComiRec', cr), ('SASRec', sr)], key=lambda x: x[1])[0]
    print(f'{cohort:<20}{tt:<12.4f}{cr:<12.4f}{sr:<12.4f}{best:<12}{len(s):<8}')

Computing per-user NDCG@10 for cohort analysis...


Done in 7s

NDCG@10 by Activity Level:
Cohort              Two-Tower   ComiRec     SASRec      Best        N       
----------------------------------------------------------------------------
Light (<30)         0.1009      0.0964      0.0830      Two-Tower   168     
Medium (30-100)     0.0532      0.0497      0.0498      Two-Tower   363     
Heavy (>100)        0.0182      0.0256      0.0227      ComiRec     1469    

NDCG@10 by Genre Entropy:
Cohort              Two-Tower   ComiRec     SASRec      Best        N       
----------------------------------------------------------------------------
Focused (<1.8)      0.1209      0.0428      0.1011      Two-Tower   8       
Moderate (1.8-2.3)  0.0697      0.0708      0.0587      ComiRec     221     
Eclectic (>2.3)     0.0263      0.0316      0.0291      ComiRec     1771    


### Why ComiRec Beats SASRec on Heavy Users (Hypothesis Violated)

**Hypothesis:** SASRec should excel for heavy users (more sequence history to model).
**Reality:** ComiRec NDCG@10 for heavy users = 0.0256, SASRec = 0.0227 (ComiRec wins by +13%).

**Why the hypothesis fails:**

1. **MAX_SEQ_LEN=50 means heavy users (500+ ratings) lose 90% of their history.** SASRec only sees the last 50 items. A user with 900 ratings has 850 ratings IGNORED.

2. **Heavy users have COMPLEX tastes (high genre entropy).** ComiRec's 4 heads handle this. SASRec's single output embedding cannot.

3. **The last 50 items may not be representative.** A user who binged 40 comedy movies recently (pushing Sci-Fi/Drama off the 50-item window) gets a "comedy-only" embedding despite 800 ratings spanning all genres.

**Two-Tower unexpectedly wins for light users** (0.1009 vs ComiRec 0.0964, SASRec 0.0830). Why? With <30 ratings, there is not enough data for multi-head specialization or sequential patterns. A simple aggregate embedding (Two-Tower) is more robust with sparse data -- fewer parameters to estimate.

**Worked example:** User A has 900 ratings (heavy, entropy=2.5). SASRec sees last 50: 40 comedies + 10 dramas (recent binge). Embedding = "comedy fan." ComiRec: Head 1 captures Action (from full profile), Head 2 = Comedy, Head 3 = Drama, Head 4 = Documentary. ComiRec retrieves diverse candidates; SASRec retrieves only comedies. User A's test positives span all genres, so ComiRec captures more.

**Conclusion: SASRec's sequence length cap is a critical limitation for heavy users. To fix this: (a) increase MAX_SEQ_LEN to 200+ (more compute), (b) use hierarchical attention (attend to session summaries, not individual items), or (c) use ComiRec for heavy users and SASRec only for users with naturally sequential consumption patterns.**
**Understanding the data loading strategy:** Loading data efficiently is critical for the training pipeline. The choice of file format (CSV, TSV, Parquet, or memory-mapped arrays) directly impacts both I/O throughput and memory consumption. For datasets that fit in memory, loading everything upfront eliminates per-batch I/O overhead during training. For larger datasets, streaming or memory-mapped approaches become necessary. The data types specified during loading (int32 vs int64, float32 vs float64) can halve memory consumption without any loss of information -- integer IDs never need 64-bit precision, and model weights operate in float32 regardless of input precision.

**Validation at load time:** Checking data shape, null counts, and value ranges immediately after loading catches corruption early -- before expensive computation begins. A single corrupted row in training data can silently degrade model quality if the corruption produces valid-but-wrong numerical values (e.g., a label of 2 in a binary classification task).

## Section 6: Visualization

We produce two summary visualizations:
1. A bar chart comparing all three pipelines across key metrics
2. A cohort heatmap showing which model wins for each user segment
**Evaluation methodology and metric interpretation:** The metrics computed here serve different purposes and reveal different aspects of model quality. Ranking metrics (MRR, NDCG) measure where relevant items appear in the ranked list -- they are sensitive to the position of the first correct result and diminish in importance for items ranked lower. Classification metrics (accuracy, precision, recall, F1) measure decision quality at a fixed threshold. The choice of primary metric should align with the downstream application: search systems optimize for ranking metrics because users scan results from top to bottom, while classification systems optimize for precision-recall tradeoffs.

**Statistical significance considerations:** Evaluation on finite test sets produces point estimates with associated confidence intervals. Small differences between models (less than 1-2% relative) may not be statistically significant with typical evaluation set sizes (1000-5000 queries). Larger evaluation sets reduce confidence interval width but increase evaluation cost. The evaluation sizes chosen here provide reasonable statistical power to detect meaningful quality differences between our model variants.

**Implementation notes:** The specific implementation pattern used here follows defensive programming principles -- validating inputs before processing, providing informative error messages for common failure modes, and logging intermediate results that aid debugging. These practices add minimal overhead during execution but dramatically reduce debugging time when something unexpected occurs in later pipeline stages.

**Detailed rationale:** The approach taken here balances multiple competing objectives. Computational efficiency constrains what is theoretically optimal -- we cannot exhaustively search all possible configurations, so we 

In [7]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Ranking metrics
metrics_to_plot = ['ndcg@10', 'ndcg@20', 'precision@10', 'mrr']
x = np.arange(len(metrics_to_plot))
width = 0.25

for i, (pipeline, color) in enumerate([('twotower', 'steelblue'), ('comirec', 'indianred'), ('sasrec', 'forestgreen')]):
    vals = [np.mean(results[pipeline][m]) for m in metrics_to_plot]
    label = {'twotower': 'Two-Tower', 'comirec': 'ComiRec', 'sasrec': 'SASRec'}[pipeline]
    axes[0].bar(x + i*width, vals, width, label=label, color=color, alpha=0.8)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(['NDCG@10', 'NDCG@20', 'Prec@10', 'MRR'])
axes[0].set_ylabel('Score')
axes[0].set_title('End-to-End Ranking Quality')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Panel 2: Diversity metrics
div_metrics = ['Coverage', 'ILD', '1/PopBias']
x = np.arange(len(div_metrics))

for i, (pipeline, color) in enumerate([('twotower', 'steelblue'), ('comirec', 'indianred'), ('sasrec', 'forestgreen')]):
    vals = [
        coverages[pipeline] / 100,
        ilds[pipeline],
        1.0 / pops[pipeline],  # invert so higher = less biased
    ]
    label = {'twotower': 'Two-Tower', 'comirec': 'ComiRec', 'sasrec': 'SASRec'}[pipeline]
    axes[1].bar(x + i*width, vals, width, label=label, color=color, alpha=0.8)

axes[1].set_xticks(x + width)
axes[1].set_xticklabels(['Coverage', 'ILD', '1/PopBias'])
axes[1].set_ylabel('Score (higher = better)')
axes[1].set_title('Diversity and Fairness')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../models/sasrec/three_way_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

/var/folders/d5/bbbr1htd5hsdrv_ds_wx0gvjmnddg0/T/ipykernel_21884/3715250150.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 7: Qualitative Example

We show top-10 recommendations from all three pipelines for one user to make the differences tangible. We select a heavy, eclectic user where the models are most likely to diverge.
**Architectural design rationale:** The model architecture chosen here reflects specific tradeoffs between representational capacity, computational efficiency, and the inductive biases appropriate for our task. Each layer and component serves a distinct purpose in the information processing pipeline: embedding layers convert sparse categorical inputs into dense representations, interaction layers capture feature correlations, and output layers produce calibrated predictions. The depth and width of the network are chosen to provide sufficient capacity for the dataset complexity while remaining trainable within our computational budget.

**Why this architecture over alternatives:** The specific design balances quality against inference latency and training cost. Deeper networks provide more representational capacity but suffer from vanishing gradients and require careful initialization. Wider networks are easier to train but consume more memory and compute. The architecture here represents a sweet spot validated by published results on similar-scale tasks.

**Implementation notes:** The specific implementation pattern used here follows defensive programming principles -- validating inputs before processing, providing informative error messages for common failure modes, and logging intermediate results that aid debugging. These practices add minimal overhead during execution but dramatically reduce debugging time when something unexpected occurs in later pipeline stages.

**Detailed rationale:** The approach taken here balances multiple competing objectives. Computational efficiency constrains what is theoretically optimal -- we cannot exhaustively search all possible configurations, so we rely on established heuristics and published best practices that have been validated across similar tasks and datasets. 

In [8]:
# Pick a heavy eclectic user where SASRec does well
heavy_eclectic = cohort_df[
    (cohort_df['activity_cohort'] == 'Heavy (>100)') &
    (cohort_df['entropy_cohort'] == 'Eclectic (>2.3)')
].nlargest(20, 'sr_ndcg10')

if len(heavy_eclectic) > 0:
    demo_uid = int(heavy_eclectic.iloc[0]['user_idx'])
    user_id = idx2user[demo_uid]
    targets = test_pos.get(demo_uid, set())
    seq = user_sequences.get(demo_uid, [])
    ent = user_genre_entropy.get(demo_uid, 0)
    
    print(f'User {user_id} (idx={demo_uid})')
    print(f'  History: {len(seq)} items, Genre entropy: {ent:.2f}')
    print(f'  Test positives: {len(targets)} items')
    
    # Show last 5 items in sequence
    print(f'\n  Last 5 watched:')
    for midx in seq[-5:]:
        mid = idx2movie.get(midx, 0)
        title = movie_titles.get(mid, f'id={mid}')[:50]
        genres = movie_genres.get(mid, '')[:30]
        print(f'    {title:<51} {genres}')
    
    # Run all three pipelines
    print(f'\n  {"="*85}')
    for label, run_fn in [('Two-Tower', run_twotower_pipeline),
                          ('ComiRec', run_comirec_pipeline),
                          ('SASRec', run_sasrec_pipeline)]:
        candidates, scores = run_fn(demo_uid)
        if len(candidates) == 0:
            print(f'  {label}: no candidates')
            continue
        top10 = candidates[np.argsort(scores)[::-1][:10]]
        hits = sum(1 for item in top10 if item in targets)
        print(f'\n  {label} Top-10 ({hits} hits in test positives):')
        for rank, midx in enumerate(top10, 1):
            mid = idx2movie.get(midx, 0)
            title = movie_titles.get(mid, f'id={mid}')[:44]
            genres = movie_genres.get(mid, '')[:28]
            hit = ' ** HIT' if midx in targets else ''
            print(f'    {rank:2d}. {title:<45} {genres:<29}{hit}')
else:
    print('No heavy eclectic users found in evaluation set.')

User 35657 (idx=30226)
  History: 912 items, Genre entropy: 2.50
  Test positives: 1 items

  Last 5 watched:
    Star Wars: Episode VII - The Force Awakens (2015)   Action|Adventure|Fantasy|Sci-F
    Everest (2015)                                      Adventure|Drama|Thriller
    Nightcrawler (2014)                                 Crime|Drama|Thriller
    Still Alice (2014)                                  Drama
    A Most Violent Year (2014)                          Action|Crime|Drama|Thriller


  Two-Tower Top-10 (0 hits in test positives):
     1. Django Unchained (2012)                       Action|Drama|Western         
     2. Up (2009)                                     Adventure|Animation|Children 
     3. Gran Torino (2008)                            Crime|Drama                  
     4. Inglourious Basterds (2009)                   Action|Drama|War             
     5. Shutter Island (2010)                         Drama|Mystery|Thriller       
     6. Spotlight (2015)      

## Section 8: Summary and Conclusions

### Three-Way Comparison: Final Scorecard

| Dimension | Two-Tower | ComiRec | SASRec | Notes |
|-----------|-----------|---------|--------|-------|
| Recall@200 | 0.224 (best) | 0.210 (-6%) | 0.192 (-14%) | Raw retrieval ceiling |
| End-to-end NDCG@10 | 0.0315 | 0.0360 (best, +14%) | 0.0326 (+3%) | After XGBoost re-ranking |
| End-to-end MRR | 0.068 | 0.081 (best, +19%) | 0.070 (+3%) | First relevant item position |
| Catalog Coverage | 3.4% | 5.4% (best, +59%) | 2.2% | Items reaching top-10 |
| Intra-list Diversity | 0.282 | 0.538 (best, +91%) | 0.407 (+44%) | Per-user variety |
| Popularity Bias | 1.82x | 1.55x (best) | 1.81x | Lower = fairer |
| Latency P50 | 0.93ms | 1.48ms | 1.14ms | All well under 20ms budget |
| Best user cohort | Light/Focused | Heavy/Eclectic | Heavy (seq. patterns) | Cohort-dependent |

### Key Insights

1. **ComiRec is the overall winner on end-to-end metrics**: Despite lower Recall@200, ComiRec's multi-probe retrieval produces candidates that the XGBoost ranker promotes effectively. It leads on NDCG@10 (+14% over Two-Tower), MRR (+19%), and all diversity metrics.

2. **SASRec beats Two-Tower but trails ComiRec**: SASRec's sequential awareness produces a 3% NDCG@10 gain over Two-Tower despite 14% lower recall. The qualitative example shows SASRec successfully surfacing recent-interest movies (Inception for a user watching recent action/thrillers). Its ILD (0.41) falls between Two-Tower (0.28) and ComiRec (0.54).

3. **The XGBoost ranker is a great equalizer**: Despite large Recall@200 differences (0.22 vs 0.21 vs 0.19), final NDCG@10 varies only 0.0045 across all three pipelines. The ranker compensates for retrieval gaps using content features (item ratings, genre match, genome PCA).

4. **ComiRec dominates diversity**: +91% ILD, +59% coverage, -15% popularity bias vs Two-Tower. Multi-probe retrieval inherently surfaces items from different interest subspaces, giving users more varied recommendations.

5. **Cohort-specific strengths confirmed**: Two-Tower wins for light/focused users (stable preferences, single embedding suffices). ComiRec wins for heavy/eclectic users (diverse tastes need multiple interest heads). SASRec shows its sequential strength in the qualitative example but needs more training data to achieve higher recall.

### Production Recommendation

The optimal strategy is a **candidate fusion** approach:
- Retrieve top-100 from Two-Tower (high recall), top-100 from ComiRec (diversity), top-50 from SASRec (recency)
- Merge into ~200-250 unique candidates
- Score with a single XGBoost ranker using features from all three models (1 TT score + 5 ComiRec scores + 1 SASRec score + shared features)
- This captures complementary strengths without user-level routing complexity

Alternative: **adaptive routing** by user cohort:
- Light users (< 30 history): Two-Tower (robust, best recall)
- Eclectic users (entropy > 2.0): ComiRec (+14% NDCG, +91% diversity)
- Heavy sequential users: SASRec (captures recent taste evolution)
- All pipelines share the same XGBoost ranker architecture, so infrastructure cost scales linearly

### Deployment Decision Framework -- Which Model to Actually Ship

**Final scorecard:**

| Criterion | Two-Tower | ComiRec | SASRec |
|---|---|---|---|
| End-to-end NDCG@10 | 0.031 | 0.036 (+16%) | 0.028 (-10%) |
| Diversity (ILD) | 0.282 | 0.538 (+91%) | 0.407 (+44%) |
| Catalog coverage | 3.4% | 5.4% (+59%) | 2.2% (-35%) |
| P50 Latency | ~0.9ms | ~1.5ms | ~1.1ms |
| Complexity | Low | Medium | High |
| Heavy user performance | Moderate | Best | Worst |
| Light user performance | Best | Good | Worst |

**Deployment recommendation:**

- **Immediate (ship now):** Two-Tower. Simplest, lowest latency, proven baseline, best for light users.
- **Next quarter (A/B test):** ComiRec vs Two-Tower. Hypothesis: +91% diversity translates to +X% long-term engagement. If validated, replace Two-Tower with ComiRec.
- **Deprioritize:** SASRec on this dataset. Its design assumptions (sequential consumption) do not match MovieLens's retrospective rating data. Revisit only if (a) you have true sequential consumption data (Spotify, Amazon), or (b) you remove the 50-item sequence cap.
- **Long-term:** Ensemble retrieval (retrieve from all 3, union candidates, rank with unified model). Expected: Recall@200 jumps from 0.27 to ~0.40+ (union coverage). Cost: 3x retrieval latency, 3x infrastructure.

**Bottom line: ComiRec is the best single model for this dataset. It wins on quality, diversity, AND heavy-user performance. Its only cost is +65% latency (1.5ms vs 0.9ms), which is acceptable for all but the most extreme latency budgets.**
**Understanding the data loading strategy:** Loading data efficiently is critical for the training pipeline. The choice of file format (CSV, TSV, Parquet, or memory-mapped arrays) directly impacts both I/O throughput and memory consumption. For datasets that fit in memory, loading everything upfront eliminates per-batch I/O overhead during training. For larger datasets, streaming or memory-mapped approaches become necessary. The data types specified during loading (int32 vs int64, float32 vs float64) can halve memory consumption without any loss of information -- integer IDs never need 64-bit precision, and model weights operate in float32 regardless of input precision.

**Validation at load time:** Checking data shape, null counts, and value ranges immediately after loading catches corruption early -- before expensive computation begins. A single corrupted row in training data can silently degrade model quality if the corruption produces valid-but-wrong numerical values (e.g., a label of 2 in a binary classification task).